# LLMBackend Flanagan Graph Queries

This notebook resolves one pickup instruction, projects the Flanagan motion-phase sidecar into `HypothesisGraph`, then inspects the result through the current **family/view** API:

- `FlanaganFamily.make_manager(...)` wires projector orchestration into `LLMBackend`
- `FlanaganFamily.make_view(...)` exposes typed Flanagan query domains over the generic graph
- EQL queries operate on normal node domains like `MotionPlanHypothesisNode` and `MotionPhaseHypothesisNode`
- graph-native helpers such as `subgraph_for_action(...)`, `edge_exists(...)`, and DOT visualization remain available


In [ ]:
from uniworld import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, robot_view, context = load_pr2_apartment_world()
context.evaluate_conditions = False

symbol_type = Body
print("World loaded:", type(world).__name__)
print("Robot:", robot_view)


In [ ]:
from dotenv import load_dotenv
from llmr.reasoning.llm_provider import make_llm, LLMProvider

load_dotenv("../.env")

llm = make_llm(LLMProvider.OPENAI, model="gpt-4o-mini", temperature=0.0)
print("LLM ready:", getattr(llm, "model_name", llm))


## Flanagan hypothesis-graph setup

The next cell creates a fresh `HypothesisGraph`, then uses the current family abstraction to build both sides of the workflow:

- `FlanaganFamily.make_manager(graph)` for automatic projection during backend evaluation
- `FlanaganFamily.make_view(graph)` for Flanagan-specific query access over the generic graph


In [ ]:
from krrood.entity_query_language.query.match import Match
from llmr.backend import LLMBackend
from llmr.hypotheses import (
    AboutActionEdge,
    EvokesMotionPlanEdge,
    FlanaganFamily,
    FlanaganGraphView,
    HasMotionPhaseEdge,
    HypothesisGraph,
    MotionPhaseHypothesisNode,
    MotionPlanHypothesisNode,
    ProducedClaimEdge,
)
from llmr.reasoning.flanagan_reasoner import FlanaganReasoner
from pycram.datastructures.grasp import GraspDescription
from pycram.robot_plans.actions.core.pick_up import PickUpAction

graph = HypothesisGraph()
manager = FlanaganFamily.make_manager(graph)
flanagan = FlanaganFamily.make_view(graph)

INSTRUCTION = "pick up the milk from the table"

def fresh_pickup_match():
    return Match(PickUpAction)(
        object_designator=...,
        arm=...,
        grasp_description=Match(GraspDescription)(
            approach_direction=...,
            vertical_alignment=...,
            manipulator=...,
        ),
    )

def run_instruction(instruction: str):
    backend = LLMBackend(
        llm=llm,
        symbol_type=symbol_type,
        instruction=instruction,
        strict_required=True,
        reasoners=[FlanaganReasoner(llm=llm)],
        hypothesis_graph_manager=manager,
    )
    action = next(iter(backend.evaluate(fresh_pickup_match())))
    return action, backend

def phase_rows(phases):
    return [
        {
            "display_id": phase.display_id,
            "index": phase.phase_index,
            "phase": phase.phase_name,
            "target": phase.target_object,
            "contact": phase.contact,
            "motion_type": phase.motion_type,
            "urgency": phase.urgency,
            "run_id": phase.meta.short_run_id,
        }
        for phase in phases
    ]

print("Graph and helpers ready.")


In [ ]:
action, backend = run_instruction(INSTRUCTION)
current_plan = flanagan.plans()[-1]
current_run_id = current_plan.meta.run_id
action_view = flanagan.action_subgraph(action)
run_view = flanagan.run_subgraph(current_run_id)
action_graph = action_view.graph
run_graph = run_view.graph

print("Resolved action:", type(action).__name__)
print("Resolved object:", action.object_designator)
print("Resolved arm:", action.arm)
print("Run id:", current_run_id)
print("Short run id:", current_plan.meta.short_run_id)
print("Current plan id:", current_plan.display_id)
print("Full graph:", graph.node_count, "nodes /", graph.edge_count, "edges")
print("Action subgraph:", action_graph.node_count, "nodes /", action_graph.edge_count, "edges")
print("Run subgraph:", run_graph.node_count, "nodes /", run_graph.edge_count, "edges")


In [ ]:
# backend.semantics.motion_phases
import json

print(json.dumps(backend.semantics.motion_phases.model_dump(), indent=2, default=str))


## Graph-native inspection

Before using EQL, it is useful to inspect the action-local `FlanaganGraphView` directly. These helpers come from the family view, while `edge_exists(...)` and DOT export come from the generic graph.


In [ ]:
action_view.plans()


In [ ]:
phase_rows(action_view.phases())


In [ ]:
current_plan = action_view.plans()[0]
{
    "phases_for_plan": [phase.display_id for phase in action_view.phases_for_plan(current_plan)],
    "contact_phases": [phase.display_id for phase in action_view.contact_phases()],
    "phases_with_failures": [phase.display_id for phase in action_view.phases_with_failures()],
    "high_urgency_phases": [phase.display_id for phase in action_view.high_urgency_phases()],
    "has_plan_to_phase_edge": action_graph.edge_exists(HasMotionPhaseEdge, current_plan.id, action_view.phases()[0].id),
}


## Query setup

These EQL variables use the action-local `FlanaganGraphView` as their domain source. That keeps the graph core generic while still giving this notebook typed Flanagan access. The later queries still join through typed edges like `HasMotionPhaseEdge`.


In [ ]:
from krrood.entity_query_language.factories import an, entity, variable

plan = variable(MotionPlanHypothesisNode, domain=action_view.plans())
phase = variable(MotionPhaseHypothesisNode, domain=action_view.phases())
has_phase = variable(HasMotionPhaseEdge, domain=action_graph.edge_domain(HasMotionPhaseEdge))

{
    "plans": [item.display_id for item in action_view.plans()],
    "phases": [item.display_id for item in action_view.phases()],
}


In [ ]:
# Query 1: motion plan for this instruction
plan_query = an(
    entity(plan).where(
        plan.instruction_text == INSTRUCTION,
    )
)
list(plan_query.evaluate())


In [ ]:
# Query 2: phase claims attached to the current plan through HasMotionPhaseEdge
plan_id = action_view.plans()[0].id

phase_query = an(
    entity(phase).where(
        has_phase.src_id == plan_id,
        has_phase.dst_id == phase.id,
    )
)
phase_rows(list(phase_query.evaluate()))


In [ ]:
# Query 3: contact phases
contact_query = an(
    entity(phase).where(
        phase.contact == True,
    )
)
phase_rows(list(contact_query.evaluate()))


In [ ]:
# Query 4: phases with explicit failure handling
failure_query = an(
    entity(phase).where(
        phase.urgency == "high",
    )
)
phase_rows(list(failure_query.evaluate()))


## Visualization

The generic graph still supports DOT export. Rendering the action-local subgraph keeps the image readable while preserving the full typed node and edge structure.


In [ ]:
from graphviz import Source
from IPython.display import display

try:
    display(Source(action_graph.to_dot(), format="svg"))
except Exception:
    print(action_graph.to_dot())
